# Closures & Iterators

Welcome! Closures and iterators are two of Rust's most powerful features. **Closures** are anonymous functions that can capture their environment. **Iterators** let you process sequences of values in a lazy, composable way. Together they form the backbone of idiomatic Rust.

## What Are Closures and Why Do They Matter?

Closures are like "mini functions" you can define inline. They can capture variables from the surrounding scope — something regular functions cannot do. Use them for callbacks, sorting, filtering, and any place you need flexible, one-off logic.

## Closure Syntax

The basic form is `|parameters| body`. Parameters go between pipes; the body can be an expression (no braces) or a block.

In [ ]:
let add_one = |x| x + 1;
println!("add_one(5) = {}", add_one(5));

let add = |a, b| a + b;
println!("add(3, 4) = {}", add(3, 4));

let greet = |name: &str| {
    let msg = format!("Hello, {}!", name);
    msg
};
println!("{}", greet("Rust"));

## Capturing Variables: By Ref, By Mut Ref, By Value

Closures capture variables from the environment. They can capture by **reference** (borrow), **mutable reference** (borrow mutably), or **value** (move ownership). The compiler infers the capture mode from how you use the variable.

In [ ]:
let x = 10;
let by_ref = || println!("x = {}", x);  // borrows x immutably
by_ref();
println!("x still available: {}", x);

In [ ]:
let mut count = 0;
let mut inc = || {
    count += 1;  // borrows count mutably
    count
};
println!("inc: {}, inc: {}", inc(), inc());

In [ ]:
let s = String::from("hello");
let by_move = || println!("{}", s);  // moves s into closure
by_move();
// by_move();  // would error: s was moved

## Fn, FnMut, FnOnce Traits

Closures implement one or more of these traits:
- **FnOnce**: can be called once (may consume captured values)
- **FnMut**: can be called multiple times, may mutate captured values
- **Fn**: can be called multiple times, doesn't mutate (immutable borrow)

In [ ]:
// Fn: immutable borrow
let x = 5;
let f = || x + 1;
println!("{}, {}", f(), f());

// FnOnce: consumes
let s = String::from("hi");
let g = || s;  // moves s
let _ = g();   // can only call once

## Closures as Function Parameters

Pass closures to functions using generics with trait bounds: `F: Fn(T) -> U` or `impl Fn(T) -> U`.

In [ ]:
fn apply_twice<F>(f: F, x: i32) -> i32
where
    F: Fn(i32) -> i32,
{
    f(f(x))
}

let result = apply_twice(|x| x * 2, 5);
println!("apply_twice (x*2) on 5: {}", result);  // 5 -> 10 -> 20

## Returning Closures (impl Fn)

To return a closure, use `impl Fn(...) -> ...`. The closure must be boxed or returned in a way the compiler can handle.

In [ ]:
fn make_adder(n: i32) -> impl Fn(i32) -> i32 {
    move |x| x + n  // move captures n by value
}

let add_10 = make_adder(10);
println!("add_10(3) = {}", add_10(3));

## The Iterator Trait and next()

An iterator produces a sequence of values. The `Iterator` trait requires implementing `next()` which returns `Option<Item>`. When `None` is returned, iteration stops.

In [ ]:
let v = vec![1, 2, 3];
let mut iter = v.iter();
println!("next: {:?}", iter.next());
println!("next: {:?}", iter.next());
println!("next: {:?}", iter.next());
println!("next: {:?}", iter.next());  // None

## Iterator Adaptors: map, filter, enumerate, zip, chain

**Adaptors** transform one iterator into another. They're lazy — nothing runs until you consume the iterator.

In [ ]:
let v = vec![1, 2, 3, 4, 5];
let doubled: Vec<i32> = v.iter().map(|x| x * 2).collect();
println!("map: {:?}", doubled);

In [ ]:
let evens: Vec<&i32> = v.iter().filter(|x| **x % 2 == 0).collect();
println!("filter: {:?}", evens);

In [ ]:
for (i, val) in v.iter().enumerate() {
    println!("  {}: {}", i, val);
}

In [ ]:
let a = [1, 2, 3];
let b = [10, 20, 30];
let zipped: Vec<_> = a.iter().zip(b.iter()).collect();
println!("zip: {:?}", zipped);

## Consuming Adaptors: collect, sum, fold, any, all

**Consumers** drive the iterator and produce a final value. They "use up" the iterator.

In [ ]:
let nums = vec![1, 2, 3, 4, 5];
let sum: i32 = nums.iter().sum();
let product: i32 = nums.iter().fold(1, |acc, x| acc * x);
let any_gt_4 = nums.iter().any(|x| *x > 4);
let all_positive = nums.iter().all(|x| *x > 0);
println!("sum: {}, product: {}, any>4: {}, all>0: {}", sum, product, any_gt_4, all_positive);

## Lazy Evaluation

Iterators are **lazy**: adaptors don't run until a consumer pulls values. This means you can build long chains without allocating intermediate collections.

In [ ]:
let result = (1..100)
    .filter(|n| n % 2 == 0)
    .map(|n| n * n)
    .take(5)
    .sum::<i32>();
println!("sum of first 5 even squares (1..100): {}", result);

## Creating Custom Iterators

Implement the `Iterator` trait for your type. You only need to implement `next()`; other methods come for free.

In [ ]:
struct Counter {
    count: u32,
    max: u32,
}

impl Counter {
    fn new(max: u32) -> Self {
        Counter { count: 0, max }
    }
}

impl Iterator for Counter {
    type Item = u32;
    fn next(&mut self) -> Option<Self::Item> {
        if self.count < self.max {
            self.count += 1;
            Some(self.count)
        } else {
            None
        }
    }
}

let c: Vec<u32> = Counter::new(5).collect();
println!("Counter: {:?}", c);

## for Loops as Iterator Sugar

A `for` loop is syntactic sugar for consuming an iterator. `for x in iter` desugars to using `into_iter()` and calling `next()` in a loop.

In [ ]:
let v = vec!["a", "b", "c"];
for s in &v {
    println!("{}", s);
}
// Equivalent to: for s in v.iter() { ... }

## Common Mistakes Beginners Make

1. **Forgetting `move`** when returning a closure that captures by value
2. **Double dereference in filter** — `filter(|x| **x % 2 == 0)` for `&i32`
3. **Assuming iterators run immediately** — they're lazy until consumed
4. **Calling a closure that moved values more than once** — FnOnce!
5. **Using `iter()` when you need `into_iter()`** — ownership matters

## Key Rules to Remember

- Closures: `|params| body`, capture by ref/mut ref/value
- Fn / FnMut / FnOnce describe how often and how a closure can be called
- Iterators: `next()` returns `Option<Item>`, lazy by default
- Adaptors (map, filter) transform; consumers (collect, sum) drive
- `for` loops consume iterators; prefer iterators over manual loops